# Jogos do Dia — FlashScore (autossuficiente)

Este notebook coleta os jogos de hoje, amanhã e depois de amanhã sem depender de arquivos Python do projeto. Execute as células na ordem.

In [1]:
# Dependências: selenium, tinydb, tqdm e pandas
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from tinydb import Query, TinyDB
from tqdm.auto import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# ========================= CONFIGURAÇÃO =========================
DIAS = [0, 1, 2]          # 0=hoje, 1=amanhã, 2=depois de amanhã
HEADLESS = True           # False mostra a janela do Chrome
SOMENTE_FALTANTES = False # False apaga e refaz; True preserva os existentes
TEMPO_ENTRE_JOGOS = 0.4
MAX_TENTATIVAS_JOGO = 2
PASTA_SAIDA = Path('.')

# Por padrão coleta todas as ligas. Para filtrar, ative e informe os nomes
# exatamente como aparecem no FlashScore (em maiúsculas).
FILTRAR_LIGAS = False
LEAGUES_MONITORADAS = {
    'EUROPE - CHAMPIONS LEAGUE',
    'EUROPE - CONFERENCE LEAGUE',
    'EUROPE - EUROPA LEAGUE',
    'SOUTH AMERICA - COPA LIBERTADORES',
    'SOUTH AMERICA - COPA SUDAMERICANA',
    'BRAZIL - SERIE A BETANO',
    'BRAZIL - SERIE B SUPERBET',
}

FOOTBALL_URL = 'https://www.flashscore.com/football/'
BASE_URL = 'https://www.flashscore.com'


/home/leandrofilho/GitHub/Fabio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_fast_options(headless=True):
    options = Options()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-notifications')
    options.add_argument('--disable-popup-blocking')
    options.add_argument('--lang=en-US')
    options.page_load_strategy = 'eager'
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('prefs', {
        'profile.managed_default_content_settings.images': 2,
        'profile.default_content_setting_values.notifications': 2,
    })
    return options


def init_driver(headless=True):
    options = build_fast_options(headless)
    # Usa um driver local se existir; caso contrário, o Selenium Manager
    # encontra/baixa automaticamente a versão compatível com o Chrome.
    candidates = [Path('chromedriver.exe'), Path('chromedriver')]
    local_driver = next((p for p in candidates if p.is_file()), None)
    if local_driver:
        driver = webdriver.Chrome(service=Service(str(local_driver.resolve())), options=options)
    else:
        driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(40)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def accept_flashscore_consents(driver, timeout=5):
    accepted_cookies = False
    accepted_age = False
    try:
        button = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.ID, 'onetrust-accept-btn-handler'))
        )
        driver.execute_script('arguments[0].click();', button)
        accepted_cookies = True
        time.sleep(0.5)
    except Exception:
        pass

    # O limite varia por região: atualmente aparece 18, mas já apareceu 24.
    try:
        for element in driver.find_elements(By.CSS_SELECTOR, "button, a, [role='button']"):
            text = (element.text or '').strip().lower()
            if 'and older' in text or 'or older' in text:
                driver.execute_script('arguments[0].click();', element)
                accepted_age = True
                time.sleep(1)
                break
    except Exception:
        pass

    if accepted_cookies:
        print('  ✓ Cookies aceitos')
    if accepted_age:
        print('  ✓ Confirmação de idade aceita')
    return accepted_cookies or accepted_age


def navigate(driver, url, wait_css=None, timeout=20):
    driver.get(url)
    if wait_css:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css))
        )


def go_to_future_day(driver, days_ahead):
    if days_ahead <= 0:
        return
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="wcl-dayPicker"]'))
    )
    for step in range(days_ahead):
        button = WebDriverWait(driver, 15).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-day-picker-arrow="next"]'))
        )
        old_cards = driver.find_elements(By.CSS_SELECTOR, 'div[id^="g_1_"]')
        reference = old_cards[0] if old_cards else None
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
        driver.execute_script('arguments[0].click();', button)
        if reference is not None:
            try:
                WebDriverWait(driver, 10).until(EC.staleness_of(reference))
            except Exception:
                pass
        time.sleep(1.5)
        print(f'  ✓ Navegou para +{step + 1} dia(s)')


def expand_sections(driver):
    # O nome CSS muda com frequência; o texto é uma alternativa mais estável.
    xpath = (
        "//*[self::span or self::button][contains(translate(normalize-space(.), "
        "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'display matches')]"
    )
    expanded = 0
    for button in driver.find_elements(By.XPATH, xpath):
        try:
            if button.is_displayed():
                driver.execute_script('arguments[0].click();', button)
                expanded += 1
        except Exception:
            pass
    if expanded:
        print(f'  ✓ Expandiu {expanded} seções')
        time.sleep(1)


def wait_for_match_ids(driver, timeout=25):
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, 'div[id^="g_1_"]')) > 0
        )
    except Exception:
        pass
    ids = []
    seen = set()
    for card in driver.find_elements(By.CSS_SELECTOR, 'div[id^="g_1_"]'):
        full_id = card.get_attribute('id') or ''
        match_id = full_id.split('_')[-1]
        if match_id and match_id not in seen:
            seen.add(match_id)
            ids.append(match_id)
    return ids


def _element_text(driver, selector):
    elements = driver.find_elements(By.CSS_SELECTOR, selector)
    return elements[0].text.strip() if elements else ''


def scrape_match_basic(match_id, driver, season):
    url = f'{BASE_URL}/match/{match_id}/#/match-summary/match-summary'
    navigate(driver, url, wait_css='div.duelParticipant', timeout=20)
    WebDriverWait(driver, 20).until(
        lambda d: _element_text(d, 'div.duelParticipant__home div.participant__participantName')
        and _element_text(d, 'div.duelParticipant__away div.participant__participantName')
    )

    data = {
        'Id': match_id,
        'League': '-',
        'Round': '-',
        'Date': '',
        'Time': '',
        'Home': _element_text(driver, 'div.duelParticipant__home div.participant__participantName'),
        'Away': _element_text(driver, 'div.duelParticipant__away div.participant__participantName'),
        'Season': str(season),
    }

    overlines = [
        element.text.strip()
        for element in driver.find_elements(By.CSS_SELECTOR, 'span[data-testid="wcl-scores-overline-03"]')
        if element.text.strip()
    ]
    if len(overlines) >= 3:
        parts = f'{overlines[1]} - {overlines[2]}'.split(' - ')
        if len(parts) >= 3:
            data['League'] = ' - '.join(parts[:2]).upper()
            data['Round'] = ' - '.join(parts[2:]).upper()
        else:
            data['League'] = ' - '.join(parts).upper()

    start = _element_text(driver, 'div.duelParticipant__startTime')
    start_parts = start.split()
    if start_parts:
        data['Date'] = start_parts[0].replace('.', '/')
    if len(start_parts) >= 2:
        data['Time'] = start_parts[1]
    return data


def process_day(driver, days_ahead, somente_faltantes=False):
    target = datetime.today() + timedelta(days=days_ahead)
    date_iso = target.strftime('%Y-%m-%d')
    output_path = PASTA_SAIDA / f'Jogos_Flashscore_Football_{date_iso}.json'

    print(f'\n{"=" * 72}\nColetando {date_iso} (+{days_ahead} dia(s))\n{"=" * 72}')
    navigate(driver, FOOTBALL_URL, wait_css='div#live-table', timeout=20)
    accept_flashscore_consents(driver)
    go_to_future_day(driver, days_ahead)
    expand_sections(driver)
    match_ids = wait_for_match_ids(driver)
    print(f'Jogos encontrados na página: {len(match_ids)}')

    if not match_ids:
        raise RuntimeError(
            'Nenhum cartão carregou. O arquivo anterior foi preservado; tente novamente.'
        )

    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    if not somente_faltantes and output_path.exists():
        output_path.unlink()
    db = TinyDB(output_path)
    query = Query()
    existing_ids = {row.get('Id') for row in db.all() if row.get('Id')}
    pending_ids = [match_id for match_id in match_ids if match_id not in existing_ids]
    print(f'Existentes: {len(match_ids) - len(pending_ids)} | A coletar: {len(pending_ids)}')

    saved = 0
    try:
        for match_id in tqdm(pending_ids, desc=date_iso):
            game = None
            for attempt in range(1, MAX_TENTATIVAS_JOGO + 1):
                try:
                    game = scrape_match_basic(match_id, driver, season=target.year)
                    break
                except Exception as error:
                    if attempt == MAX_TENTATIVAS_JOGO:
                        print(f'  ✗ {match_id}: {type(error).__name__}: {error}')
                    else:
                        time.sleep(1)
            if not game:
                continue
            if FILTRAR_LIGAS and game['League'] not in LEAGUES_MONITORADAS:
                continue
            if not db.search(query.Id == match_id):
                db.insert(game)
                saved += 1
                print(f"  ✓ {match_id} — {game['Home']} x {game['Away']}")
            time.sleep(TEMPO_ENTRE_JOGOS)
    finally:
        total = len(db.all())
        db.close()

    print(f'Concluído: {saved} novos | {total} no arquivo {output_path}')
    return output_path


In [ ]:
# Executa a coleta com um único Chrome (baixo consumo de memória).
driver = init_driver(headless=HEADLESS)
arquivos_gerados = []
try:
    for days_ahead in DIAS:
        try:
            path = process_day(driver, days_ahead, somente_faltantes=SOMENTE_FALTANTES)
            arquivos_gerados.append(path)
        except Exception as error:
            print(f'Falha no dia +{days_ahead}: {type(error).__name__}: {error}')
finally:
    driver.quit()

print('\nArquivos gerados:')
for path in arquivos_gerados:
    print(f'  - {path}')



Coletando 2026-07-30 (+0 dia(s))
  ✓ Cookies aceitos
  ✓ Confirmação de idade aceita
  ✓ Expandiu 15 seções
Jogos encontrados na página: 155
Existentes: 0 | A coletar: 155


2026-07-30:   0%|          | 0/155 [00:00<?, ?it/s]

  ✓ xUbm2vec — APR x Vipers


2026-07-30:   1%|          | 1/155 [00:02<06:31,  2.54s/it]

  ✓ pf2e0IP9 — Gor Mahia x Garde Republicaine


...

  ✓ 6RqpAzue — Logrono W (ESP) x Real Sociedad W (ESP)


2026-07-31: 100%|█████████▉| 405/406 [18:14<00:02,  2.42s/it]

  ✓ A3lPz2Ua — Herentals x TelOne


2026-07-31: 100%|██████████| 406/406 [18:17<00:00,  2.70s/it]


Concluído: 406 novos | 406 no arquivo Jogos_Flashscore_Football_2026-07-31.json

Coletando 2026-08-01 (+2 dia(s))


In [ ]:
# Carrega todos os arquivos gerados em um único DataFrame.
frames = []
for path in arquivos_gerados:
    db = TinyDB(path)
    try:
        frames.append(pd.DataFrame(db.all()))
    finally:
        db.close()

jogos_fabio = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if not jogos_fabio.empty:
    jogos_fabio = jogos_fabio.drop_duplicates(subset=['Id']).sort_values(['Date', 'Time'])
display(jogos_fabio)


jogos_fabio.to_excel('Jogos_Flashscore_Fabio.xlsx', index=False)
